# dots.ocr — Kaggle GPU Server

**Trước khi chạy:** Bật GPU bằng cách vào Settings (góc phải) → Accelerator → **GPU T4 x1**


## Cell 1 — Kiểm tra GPU + cài packages

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

!pip install "vllm>=0.11.0,<0.15.0" pyngrok xformers -q
import vllm
print(f"✅ vLLM {vllm.__version__} installed")

import torch
cuda_ver = torch.version.cuda.replace(".", "")
torch_ver = f"{torch.__version__.split('.')[0]}.{torch.__version__.split('.')[1]}"
wheel_url = f"https://flashinfer.ai/whl/cu{cuda_ver}/torch{torch_ver}/"
r = subprocess.run(["pip", "install", "flashinfer-python", "-i", wheel_url, "-q"],
                   capture_output=True, text=True)
print("✅ FlashInfer done" if r.returncode == 0 else f"⚠️ FlashInfer skip: {r.stderr[-80:]}")

## Cell 2 — Clone repo + tải model (~4GB, 5-10 phút)

In [ ]:
import os

# Clone repo
if not os.path.exists('/kaggle/working/dots.ocr'):
    !git clone https://github.com/hoanggiangppe-tech/dots.ocr.git /kaggle/working/dots.ocr

%cd /kaggle/working/dots.ocr
!git checkout claude/setup-local-repo-4LmIb

# Download model
if not os.path.exists('/kaggle/working/dots.ocr/weights/DotsMOCR'):
    !python3 tools/download_model.py
    print('✅ Model downloaded')
else:
    print('✅ Model đã có sẵn')

## Cell 3 — Patch model code (chạy 1 lần)

In [ ]:
import os

def patch_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    new_lines = []
    modified = False
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.lstrip()
        indent = line[:len(line) - len(stripped)]
        if any(f'Auto{t}.register' in line for t in ['Config','Model','Processor','Tokenizer']):
            if i == 0 or 'try:' not in lines[i-1]:
                new_lines += [f'{indent}try:\n', f'{indent}    {stripped}',
                              f'{indent}except (ValueError, AssertionError):\n',
                              f'{indent}    pass  # already registered\n']
                modified = True
                i += 1
                continue
        new_lines.append(line)
        i += 1
    if modified:
        with open(file_path, 'w') as f:
            f.writelines(new_lines)
    return modified

patched = []
for root, _, files in os.walk('./weights/DotsMOCR'):
    for fname in files:
        if fname.endswith('.py'):
            fpath = os.path.join(root, fname)
            if patch_file(fpath):
                patched.append(fname)
                print(f'✅ Patched: {fname}')

print(f'Done — patched {len(patched)} file(s)')

# --- Patch flash_attn import (T4 không hỗ trợ flash-attn 2) ---
FLASH_FALLBACK = """
try:
    from flash_attn import flash_attn_varlen_func
except ImportError:
    import torch, math
    def flash_attn_varlen_func(q, k, v, cu_seqlens_q, cu_seqlens_k,
                               max_seqlen_q, max_seqlen_k,
                               dropout_p=0.0, softmax_scale=None,
                               causal=False, **kwargs):
        if softmax_scale is None:
            softmax_scale = 1.0 / math.sqrt(q.shape[-1])
        outputs = []
        for b in range(len(cu_seqlens_q) - 1):
            qs, qe = cu_seqlens_q[b].item(), cu_seqlens_q[b+1].item()
            ks, ke = cu_seqlens_k[b].item(), cu_seqlens_k[b+1].item()
            qb = q[qs:qe].transpose(0,1).unsqueeze(0)
            kb = k[ks:ke].transpose(0,1).unsqueeze(0)
            vb = v[ks:ke].transpose(0,1).unsqueeze(0)
            out = torch.nn.functional.scaled_dot_product_attention(
                qb, kb, vb, scale=softmax_scale, is_causal=causal)
            outputs.append(out.squeeze(0).transpose(0,1))
        return torch.cat(outputs, dim=0)
"""

for root, _, files in os.walk('./weights/DotsMOCR'):
    for fname in files:
        if fname.endswith('.py'):
            fpath = os.path.join(root, fname)
            with open(fpath, 'r') as f:
                content = f.read()
            if 'from flash_attn import flash_attn_varlen_func' in content and 'except ImportError' not in content:
                content = content.replace(
                    'from flash_attn import flash_attn_varlen_func',
                    FLASH_FALLBACK.strip()
                )
                with open(fpath, 'w') as f:
                    f.write(content)
                print(f'✅ Patched flash_attn: {fname}')

# Xóa HuggingFace module cache để load lại file đã patch
import shutil
cache_dir = os.path.expanduser('~/.cache/huggingface/modules/transformers_modules')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print('✅ HuggingFace modules cache cleared')


## Cell 3b — Fix libcuda stub (chạy mỗi session)

In [ ]:
import subprocess, os

# --- 1. libcuda.so stub (compile-time only, mỗi session) ---
with open('/tmp/cuda_stub.c', 'w') as f:
    f.write('void _cuda_stub_(void) {}\n')
r = subprocess.run(
    ['gcc', '-shared', '-fPIC', '-o', '/tmp/libcuda.so', '/tmp/cuda_stub.c'],
    capture_output=True, text=True
)
if r.returncode == 0:
    subprocess.run(['rm', '-rf', '/root/.cache/flashinfer'], capture_output=True)
    os.environ['LIBRARY_PATH'] = '/tmp'
    print('✅ libcuda.so stub created (compile-time only), cache cleared')
else:
    print(f'❌ gcc failed: {r.stderr}')

# --- 2. Patch vLLM FlashInfer backend: SDPA fallback cho T4 sm_75 ---
# BatchPrefillWithPagedKVCache bị lỗi "invalid argument" trên T4 + CUDA 13.0
# → Wrap bằng try/except + PyTorch SDPA fallback
fi_path = '/usr/local/lib/python3.12/dist-packages/vllm/v1/attention/backends/flashinfer.py'
_MARKER = '# _SDPA_FALLBACK_PATCHED_'
try:
    with open(fi_path) as f:
        content = f.read()

    if _MARKER in content:
        print('✅ flashinfer.py already patched')
    else:
        old_block = (
            '                    prefill_wrapper.run(\n'
            '                        prefill_query,\n'
            '                        kv_cache_permute,\n'
            '                        k_scale=layer._k_scale_float,\n'
            '                        v_scale=layer._v_scale_float,\n'
            '                        out=out_prefill,\n'
            '                        kv_cache_sf=kv_cache_sf,\n'
            '                    )'
        )
        new_block = (
            '                    ' + _MARKER + '\n'
            '                    try:\n'
            '                        prefill_wrapper.run(\n'
            '                            prefill_query,\n'
            '                            kv_cache_permute,\n'
            '                            k_scale=layer._k_scale_float,\n'
            '                            v_scale=layer._v_scale_float,\n'
            '                            out=out_prefill,\n'
            '                            kv_cache_sf=kv_cache_sf,\n'
            '                        )\n'
            '                    except Exception:\n'
            '                        import torch.nn.functional as _F\n'
            '                        _q = prefill_query.float()\n'
            '                        _k = key[num_decode_tokens:].float()\n'
            '                        _v = value[num_decode_tokens:].float()\n'
            '                        _q_t = _q.transpose(0, 1).unsqueeze(0)\n'
            '                        _k_t = _k.transpose(0, 1).unsqueeze(0)\n'
            '                        _v_t = _v.transpose(0, 1).unsqueeze(0)\n'
            '                        _nh, _nkvh = _q_t.shape[1], _k_t.shape[1]\n'
            '                        if _nh != _nkvh:\n'
            '                            _k_t = _k_t.repeat_interleave(_nh // _nkvh, dim=1)\n'
            '                            _v_t = _v_t.repeat_interleave(_nh // _nkvh, dim=1)\n'
            '                        _out = _F.scaled_dot_product_attention(\n'
            '                            _q_t, _k_t, _v_t,\n'
            '                            is_causal=True, scale=self.scale)\n'
            '                        out_prefill.copy_(\n'
            '                            _out.squeeze(0).transpose(0, 1).to(out_prefill.dtype))'
        )

        if old_block in content:
            content = content.replace(old_block, new_block)
            with open(fi_path, 'w') as f:
                f.write(content)
            print('✅ flashinfer.py patched — SDPA fallback added (T4/sm_75)')
        else:
            print('❌ Block không tìm thấy — in debug:')
            lines = content.split('\n')
            for i, l in enumerate(lines[1570:1582], 1571):
                print(f'{i}: {repr(l)}')
except FileNotFoundError:
    print(f'⚠️ Không tìm thấy {fi_path} — bỏ qua patch')

## Cell 4a — Tạo ngrok tunnel (chạy 1 lần/session, KHÔNG chạy lại khi vLLM crash)

In [ ]:
import os
from pyngrok import ngrok, conf

# Lấy token tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""  # <-- dán token vào đây

if not NGROK_TOKEN:
    print('⚠️ Chưa có token! Dán token vào NGROK_TOKEN = "..." rồi chạy lại')
else:
    conf.get_default().auth_token = NGROK_TOKEN

    ngrok.kill()
    import time; time.sleep(2)
    tunnel = ngrok.connect(8000, bind_tls=True)
    public_url = tunnel.public_url

    with open('/tmp/tunnel_url.txt', 'w') as f:
        f.write(public_url)

    print('\n' + '='*60)
    print(f'🌐 URL: {public_url}')
    print('='*60)
    print('👆 Dán URL này vào ô Server URL trong UI local!')
    print('\n⚠️  Quan trọng: KHÔNG chạy lại Cell 4a khi vLLM crash!')
    print('   → Chỉ chạy lại Cell 4b — URL sẽ KHÔNG đổi.\n')

## Cell 4b — Start vLLM server (chạy lại khi vLLM crash, KHÔNG ảnh hưởng ngrok)

In [ ]:
import subprocess, time, requests, os

# Đọc URL từ Cell 4a
try:
    with open('/tmp/tunnel_url.txt') as f:
        public_url = f.read().strip()
    print(f'🔗 Dùng URL: {public_url}')
except FileNotFoundError:
    print('⚠️ Chưa có URL — chạy Cell 4a trước!')
    public_url = None

if public_url:
    env = os.environ.copy()
    env['VLLM_USE_V1'] = '0'
    env['VLLM_ATTENTION_BACKEND'] = 'XFORMERS'
    env['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
    env['LIBRARY_PATH'] = '/tmp'

    # Chỉ kill vLLM, KHÔNG kill cloudflared
    subprocess.run(['pkill', '-f', 'vllm serve'], capture_output=True)
    time.sleep(3)

    log_file = open('/tmp/vllm.log', 'w')
    proc = subprocess.Popen([
        'vllm', 'serve', './weights/DotsMOCR',
        '--tensor-parallel-size', '1',
        '--gpu-memory-utilization', '0.95',
        '--max-model-len', '32768',
        '--dtype', 'half',
        '--chat-template-content-format', 'string',
        '--served-model-name', 'model',
        '--trust-remote-code',
        '--enforce-eager',
        '--port', '8000',
    ], stdout=log_file, stderr=log_file, env=env,
        preexec_fn=os.setsid)

    print('⏳ Đang khởi động vLLM (3-5 phút lần đầu)...')
    for i in range(120):
        time.sleep(5)
        if proc.poll() is not None:
            log_file.flush()
            print(f'❌ Server crash! Exit code: {proc.poll()}')
            with open('/tmp/vllm.log') as f:
                lines = f.readlines()
            errors = [l for l in lines if any(k in l for k in ['ERROR','ValueError','RuntimeError','failed','OOM'])]
            print(''.join(errors[-20:]))
            break
        try:
            if requests.get('http://localhost:8000/v1/models', timeout=3).status_code == 200:
                print(f'\n✅ vLLM sẵn sàng sau {(i+1)*5}s!')
                print(f'🌐 URL: {public_url}')
                break
        except:
            if (i+1) % 6 == 0:
                print(f'   [{(i+1)*5}s] Đang khởi động...')
    else:
        print('⚠️ Timeout — xem log: open("/tmp/vllm.log").readlines()[-30:]')

## Cell 5 — Keep-alive (chạy sau khi vLLM sẵn sàng)

In [ ]:
import time, requests, subprocess
from pyngrok import ngrok

print('🔄 Keep-alive đang chạy — giữ cell này running để Kaggle session không timeout.')
print('   Nhấn Stop để dừng.\n')

count = 0
failures = 0

while True:
    time.sleep(60)
    count += 1

    # --- Kiểm tra vLLM ---
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=5)
        if r.status_code == 200:
            failures = 0
            print(f'   [{count:3d} phút] ✅ Server OK')
        else:
            failures += 1
            print(f'   [{count:3d} phút] ⚠️ HTTP {r.status_code} (lần {failures})')
    except Exception as e:
        failures += 1
        print(f'   [{count:3d} phút] ❌ vLLM không phản hồi (lần {failures}): {e}')
        if failures >= 3:
            print('\n⛔ vLLM đã crash sau 3 lần thử.')
            print('   → Restart vLLM: chỉ chạy lại Cell 4b (URL KHÔNG đổi!)')
            break

    # --- Kiểm tra ngrok mỗi 5 phút ---
    # QUAN TRỌNG: double-check trước khi reconnect để tránh tạo URL mới
    # trong khi người dùng đang OCR → gây ERR_NGROK_3004
    if count % 5 == 0:
        try:
            tunnels = ngrok.get_tunnels()
            if not tunnels:
                print(f'   [{count:3d} phút] ⚠️ Ngrok tunnels trống — chờ 30s rồi kiểm tra lại...')
                time.sleep(30)
                tunnels = ngrok.get_tunnels()  # double-check
                if not tunnels:
                    t = ngrok.connect(8000, bind_tls=True)
                    new_url = t.public_url
                    with open('/tmp/tunnel_url.txt', 'w') as f:
                        f.write(new_url)
                    print(f'\n🔄 [{count:3d} phút] Ngrok tunnel mất — đã reconnect!')
                    print(f'   🌐 URL MỚI: {new_url}')
                    print('   👆 Cập nhật URL này trong ô Server URL của UI local!\n')
                else:
                    print(f'   [{count:3d} phút] ✅ Ngrok OK (false alarm)')
        except Exception as e:
            print(f'   [{count:3d} phút] ⚠️ Ngrok check lỗi: {e}')

---

## ══ PHIÊN BẢN ZROK ══ (thay thế ngrok — ổn định hơn, 5GB/ngày)

> **Dùng Cell 4a_zrok thay Cell 4a** · Cell 4b dùng chung · **Dùng Cell 5_zrok thay Cell 5**
>
> Token lấy tại: https://zrok.io → đăng nhập → **Account → Enable Token**

## Cell 4a_zrok — Tạo zrok tunnel (chạy 1 lần/session, Cell 4b dùng chung)

In [ ]:
import subprocess, os, time, re, threading, pty
import json as _json

ZROK_TOKEN = ""  # <-- Token lấy tại: https://zrok.io → Account → Enable Token

ZROK_BIN = '/tmp/zrok_bin'  # tên cố định, copy từ binary trong archive

def start_zrok_share():
    """Khởi động zrok share với pseudo-TTY để tránh lỗi /dev/tty trên Jupyter"""
    open('/tmp/zrok.log', 'w').close()
    master_fd, slave_fd = pty.openpty()
    proc = subprocess.Popen(
        [ZROK_BIN, 'share', 'public', 'http://localhost:8000'],
        stdout=slave_fd, stderr=slave_fd, stdin=slave_fd,
        close_fds=True, preexec_fn=os.setsid
    )
    os.close(slave_fd)
    def _drain(fd):
        with open('/tmp/zrok.log', 'ab') as f:
            while True:
                try:
                    chunk = os.read(fd, 4096)
                    if not chunk: break
                    f.write(chunk); f.flush()
                except OSError: break
        try: os.close(fd)
        except: pass
    threading.Thread(target=_drain, args=(master_fd,), daemon=True).start()
    with open('/tmp/zrok.pid', 'w') as f:
        f.write(str(proc.pid))
    return proc

_ANSI = re.compile(rb'(?:[@-Z\-_]|\[[0-?]*[ -/]*[@-~])')

def parse_zrok_url():
    try:
        with open('/tmp/zrok.log', 'rb') as f:
            raw = f.read()
        output = _ANSI.sub(b'', raw).decode('utf-8', errors='replace')
        m = re.search(r'https://[^\s]+\.share\.zrok\.io[^\s]*', output)
        if not m:
            m = re.search(r'https://[^\s]+zrok\.io[^\s]*', output)
        return m.group(0).rstrip('/') if m else None
    except Exception:
        return None

if not ZROK_TOKEN:
    print('⚠️  Chưa có token! Dán token vào ZROK_TOKEN = "..." rồi chạy lại')
else:
    # --- 1. Tải binary nếu chưa có ---
    if not os.path.exists(ZROK_BIN):
        print('⬇️  Đang lấy URL download từ GitHub API...')
        api = subprocess.run([
            'curl', '-s',
            'https://api.github.com/repos/openziti/zrok/releases/latest'
        ], capture_output=True, text=True)

        download_url = None
        try:
            release = _json.loads(api.stdout)
            for asset in release.get('assets', []):
                if 'linux_amd64' in asset['name'] and asset['name'].endswith('.tar.gz'):
                    download_url = asset['browser_download_url']
                    print(f'   Version : {release.get("tag_name", "?")}')
                    print(f'   URL     : {download_url}')
                    break
        except Exception as e:
            print(f'❌ Parse API lỗi: {e}
   Raw: {api.stdout[:300]}')

        if not download_url:
            print('❌ Không tìm được URL từ GitHub API — dừng.')
        else:
            dl = subprocess.run(
                ['curl', '-L', '-s', '-o', '/tmp/zrok.tar.gz', download_url],
                capture_output=True, text=True
            )
            if dl.returncode != 0:
                print(f'❌ Download lỗi: {dl.stderr}')
            else:
                toc = subprocess.run(['tar', '-tzf', '/tmp/zrok.tar.gz'], capture_output=True, text=True)
                binary_name = None
                for entry in toc.stdout.strip().split('
'):
                    entry = entry.strip().lstrip('./')
                    if entry.startswith('zrok') and '/' not in entry:
                        binary_name = entry; break
                print(f'   Binary in archive: {binary_name}')
                ex = subprocess.run(['tar', '-xzf', '/tmp/zrok.tar.gz', '-C', '/tmp'], capture_output=True, text=True)
                if ex.returncode != 0:
                    print(f'❌ Giải nén lỗi: {ex.stderr}')
                elif binary_name and os.path.exists(f'/tmp/{binary_name}'):
                    subprocess.run(['cp', f'/tmp/{binary_name}', ZROK_BIN])
                    subprocess.run(['chmod', '+x', ZROK_BIN])
                    print(f'✅ zrok binary ready ({binary_name} → zrok_bin)')
                else:
                    find = subprocess.run(
                        ['find', '/tmp', '-maxdepth', '2', '-type', 'f',
                         '-name', 'zrok*', '!', '-name', '*.gz', '!', '-name', '*.tgz'],
                        capture_output=True, text=True)
                    found = [p for p in find.stdout.strip().split('
') if p]
                    if found:
                        subprocess.run(['cp', found[0], ZROK_BIN])
                        subprocess.run(['chmod', '+x', ZROK_BIN])
                        print(f'✅ zrok binary ready (fallback: {found[0]})')
                    else:
                        print('❌ Không tìm thấy binary sau giải nén')

    if not os.path.exists(ZROK_BIN):
        print('❌ Binary không tồn tại — dừng. Xem lỗi phía trên.')
    else:
        # --- 2. Enable environment ---
        print('🔑 Enabling zrok environment...')
        r = subprocess.run([ZROK_BIN, 'enable', ZROK_TOKEN], capture_output=True, text=True)
        combined = r.stdout + r.stderr
        if r.returncode == 0:
            print('✅ zrok environment enabled')
        elif 'already enabled' in combined.lower():
            print('✅ zrok đã enabled từ trước')
        else:
            print(f'❌ Enable lỗi (returncode={r.returncode}):')
            print(combined[:500])

        # --- 3. Tạo public share (pseudo-TTY để zrok v2 không lỗi /dev/tty) ---
        print('🚀 Tạo zrok public share...')
        zrok_proc = start_zrok_share()

        # --- 4. Chờ URL ---
        public_url = None
        for i in range(30):
            time.sleep(2)
            public_url = parse_zrok_url()
            if public_url: break
            if zrok_proc.poll() is not None:
                with open('/tmp/zrok.log', 'rb') as f:
                    raw = f.read()
                output = _ANSI.sub(b'', raw).decode('utf-8', errors='replace')
                print('❌ zrok process thoát sớm — xem log:')
                print(output[-800:])
                break

        if public_url:
            with open('/tmp/tunnel_url.txt', 'w') as f:
                f.write(public_url)
            print('
' + '='*60)
            print(f'🌐 URL: {public_url}')
            print('='*60)
            print('👆 Dán URL này vào ô Server URL trong UI local!')
            print('
✅ zrok: URL sẽ giữ nguyên kể cả khi vLLM crash!')
            print('   → Nếu vLLM crash, chỉ chạy lại Cell 4b
')
        else:
            with open('/tmp/zrok.log', 'rb') as f:
                raw = f.read()
            output = _ANSI.sub(b'', raw).decode('utf-8', errors='replace')
            print('❌ Không lấy được URL sau 60s — log:')
            print(output[-800:])


## Cell 5_zrok — Keep-alive (dùng với zrok, Cell 4b dùng chung)

In [ ]:
import time, requests, os, subprocess, re, threading, pty

ZROK_BIN = '/tmp/zrok_bin'  # tên cố định do Cell 4a_zrok tạo

print('🔄 Keep-alive (zrok) đang chạy.')
print('   Nhấn Stop để dừng.
')

_ANSI = re.compile(rb'(?:[@-Z\-_]|\[[0-?]*[ -/]*[@-~])')

def zrok_alive():
    try:
        with open('/tmp/zrok.pid') as f:
            pid = int(f.read().strip())
        os.kill(pid, 0)
        return True
    except:
        return False

def restart_zrok():
    open('/tmp/zrok.log', 'w').close()
    master_fd, slave_fd = pty.openpty()
    proc = subprocess.Popen(
        [ZROK_BIN, 'share', 'public', 'http://localhost:8000'],
        stdout=slave_fd, stderr=slave_fd, stdin=slave_fd,
        close_fds=True, preexec_fn=os.setsid
    )
    os.close(slave_fd)
    def _drain(fd):
        with open('/tmp/zrok.log', 'ab') as f:
            while True:
                try:
                    chunk = os.read(fd, 4096)
                    if not chunk: break
                    f.write(chunk); f.flush()
                except OSError: break
        try: os.close(fd)
        except: pass
    threading.Thread(target=_drain, args=(master_fd,), daemon=True).start()
    with open('/tmp/zrok.pid', 'w') as f:
        f.write(str(proc.pid))
    for _ in range(20):
        time.sleep(2)
        try:
            with open('/tmp/zrok.log', 'rb') as f:
                raw = f.read()
            output = _ANSI.sub(b'', raw).decode('utf-8', errors='replace')
            m = re.search(r'https://[^\s]+zrok\.io[^\s]*', output)
            if m:
                url = m.group(0).rstrip('/')
                with open('/tmp/tunnel_url.txt', 'w') as f:
                    f.write(url)
                return url
            if proc.poll() is not None:
                return None
        except Exception:
            pass
    return None

count = 0
failures = 0

while True:
    time.sleep(60)
    count += 1
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=5)
        if r.status_code == 200:
            failures = 0
            print(f'   [{count:3d} phút] ✅ Server OK')
        else:
            failures += 1
            print(f'   [{count:3d} phút] ⚠️ HTTP {r.status_code} (lần {failures})')
    except Exception as e:
        failures += 1
        print(f'   [{count:3d} phút] ❌ vLLM không phản hồi (lần {failures}): {e}')
        if failures >= 3:
            print('
⛔ vLLM đã crash. Chạy lại Cell 4b để restart.')
            break
    if count % 5 == 0:
        if not zrok_alive():
            print(f'   [{count:3d} phút] ⚠️ zrok process đã chết — đang restart...')
            new_url = restart_zrok()
            if new_url:
                print(f'   [{count:3d} phút] ✅ zrok restart OK — URL: {new_url}')
                print('   👆 Cập nhật URL mới trong ô Server URL của UI local!')
            else:
                print(f'   [{count:3d} phút] ❌ zrok restart thất bại — xem /tmp/zrok.log')
        else:
            print(f'   [{count:3d} phút] ✅ zrok OK')
